## Introduction
This notebook cleans the merged dataset before performing a chronological train-test split. Data-driven preprocessing steps—including imputation, statistical outlier detection, scaling, and encoding—are intentionally deferred until after the split to ensure that all preprocessing parameters are learned solely from the training data.

### Setup
Load the libraries and merged data and initiate a tracker to store row counts before and after each cleaning step.

In [22]:
import numpy as np
import pandas as pd

In [23]:
sold = pd.read_csv('raw-data/sold_raw.csv')
sold.shape

/var/folders/t2/p9112v_n469068__fty8_3nc0000gn/T/ipykernel_81669/106918893.py:1: DtypeWarning: Columns (0,1,4,9,74,78,79,80,82,83) have mixed types. Specify dtype option on import or set low_memory=False.
  sold = pd.read_csv('raw-data/sold_raw.csv')


(663761, 84)

In [24]:
cleaning_tracker = pd.DataFrame(columns=['Cleaning Step', 'Rows Before', 'Rows After', '% Removed'])
step_i = 0

## Part 1: Filter for single-family homes
Restrict to residential single-family properties per the project charter.

In [25]:
# confirm that all sales are closed
sold['PropertyType'].unique()

array(['Residential', 'CommercialLease', 'Land', 'ResidentialLease',
       'ManufacturedInPark', 'ResidentialIncome', 'CommercialSale',
       'BusinessOpportunity'], dtype=object)

In [26]:
sold['PropertySubType'].unique()

array(['Condominium', 'Retail', nan, 'SingleFamilyResidence', 'Duplex',
       'MultiFamily', 'Townhouse', 'Quadruplex', 'Office', 'Apartment',
       'Triplex', 'Studio', 'Loft', 'ManufacturedOnLand',
       'StockCooperative', 'Timeshare', 'Industrial', 'MixedUse',
       'Business', 'Warehouse', 'MobileHome', 'WaterPositionWithLand',
       'BoatSlip', 'Agriculture', 'Cabin', 'RoomingHouse',
       'SpecialPurpose', 'Farm', 'ManufacturedHome', 'UnimprovedLand',
       'HotelMotel', 'OwnYourOwn', 'DeededParking', 'CoOwnership',
       'Ranch'], dtype=object)

In [ ]:
sold_sfr = sold[(sold['PropertyType'] == 'Residential') & (sold['PropertySubType'] == 'SingleFamilyResidence')]
cleaning_tracker.loc[step_i] = ['Filtered for residential SFR', sold.shape[0], sold_sfr.shape[0], (1 - sold_sfr.shape[0] / sold.shape[0]) * 100]

,Cleaning Step,Rows Before,Rows After,% Removed
0,Filtered for residential SFR,663761,334704,49.574621


## Part 2: Remove features that introduce target leakage
Exclude columns that directly reveal or closely approximate the close price, reflect pricing strategy, or contain information unavailable for off-market properties. Retain date columns for subsequent record validation.

In [28]:
leakage_cols = ['OriginalListPrice', 'ListPrice', 'DaysOnMarket', 'TaxAnnualAmount',
                'ListOfficeName', 'ListAgentFullName', 'ListAgentFirstName', 'ListAgentLastName', 'ListAgentEmail', 'ListAgentAOR',
                'CoListOfficeName', 'CoListAgentFirstName', 'CoListAgentLastName', 
                'BuyerOfficeName', 'BuyerOfficeAOR', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'BuyerAgentAOR', 'BuyerAgentMlsId', 
                'BuyerAgencyCompensation', 'BuyerAgencyCompensationType', 'CoBuyerAgentFirstName']

In [29]:
sold_sfr = sold_sfr.drop(columns=leakage_cols)
sold_sfr.shape[1]

62

## Part 3: Standardize data types and formats
#### 3.1 Convert date columns to datetime

In [30]:
date_cols = ['ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate', 'CloseDate']
sold_sfr[date_cols] = sold_sfr[date_cols].apply(pd.to_datetime)

#### 3.2 Convert numeric columns to numeric

In [31]:
numeric_cols = ['ClosePrice', 'AssociationFee',
                'Latitude', 'Longitude', 
                'LivingArea', 'BathroomsTotalInteger', 'BedroomsTotal', 'MainLevelBedrooms',
                'Stories', 'GarageSpaces', 'ParkingTotal', 'FireplacesTotal',
                'LotSizeArea', 'LotSizeAcres', 'LotSizeSquareFeet', 
                'YearBuilt']
sold_sfr[numeric_cols] = sold_sfr[numeric_cols].apply(pd.to_numeric)

#### 3.3 Clean string columns
Remove leading and trailing spaces and replace empty strings with missing values.

In [32]:
string_cols = ['ListingKey', 'ListingId', 'UnparsedAddress', 'BusinessType',
               'MLSAreaMajor', 'CountyOrParish', 'SubdivisionName', 'City', 'PostalCode',
               'ElementarySchoolDistrict', 'HighSchoolDistrict', 'MiddleOrJuniorSchoolDistrict']
sold_sfr[string_cols] = sold_sfr[string_cols].apply(
    lambda col: col.astype('string').str.strip().replace('', pd.NA)
)

## Part 4: Remove redundant features
#### 4.1 Exclude high-missing columns

In [33]:
null_counts = sold_sfr.isna().sum()
null_percs =  sold_sfr.isna().mean() * 100
null_table = pd.DataFrame({
    'null count': null_counts,
    'null %': null_percs
})

Inspect the null profiles of target, physical features, temporal features, and location features.

In [34]:
null_table.loc['ClosePrice'] # 2 missing

null count    2.000000
null %        0.000598
Name: ClosePrice, dtype: float64

In [35]:
null_table.loc[['LivingArea', 'BedroomsTotal', 'BathroomsTotalInteger', 'LotSizeSquareFeet']] # 179 missing living area, 50 missing baths, 5753 missing lot size

,null count,null %
LivingArea,179,0.053480
BedroomsTotal,0,0.000000
BathroomsTotalInteger,50,0.014939
LotSizeSquareFeet,5753,1.718832


In [36]:
null_table.loc[['YearBuilt', 'ListingContractDate', 'PurchaseContractDate', 'CloseDate']] # 244 missing built year, 139 missing purchase date

,null count,null %
YearBuilt,244,0.072900
ListingContractDate,0,0.000000
PurchaseContractDate,139,0.041529
CloseDate,0,0.000000


In [37]:
null_table.loc[['MLSAreaMajor', 'CountyOrParish', 'City', 'PostalCode', 'SubdivisionName', # high missing percentages across most features
                'ElementarySchoolDistrict', 'MiddleOrJuniorSchoolDistrict', 'HighSchoolDistrict', 
                'Latitude', 'Longitude']]

,null count,null %
MLSAreaMajor,47865,14.300696
CountyOrParish,0,0.000000
City,247,0.073797
PostalCode,3,0.000896
SubdivisionName,218411,65.254972
ElementarySchoolDistrict,334704,100.000000
MiddleOrJuniorSchoolDistrict,334704,100.000000
HighSchoolDistrict,87608,26.174769
Latitude,11176,3.339070
Longitude,11176,3.339070


Remove columns with over 90% of missing values

In [38]:
high_missing_cols = sold_sfr.columns[null_percs > 90]
sold_sfr = sold_sfr.drop(columns=high_missing_cols)
sold_sfr.shape[1]

49

#### 4.2 Exclude non-informative columns
All sales should be closed, so MLS status is expected to contain only a single value. Also note that property type and subtype have become irrelevant after filtering for single-family homes.

In [39]:
sold_sfr['MlsStatus'].unique()

array(['Closed'], dtype=object)

In [40]:
sold_sfr = sold_sfr.drop(columns=['MlsStatus', 'PropertyType', 'PropertySubType'])
sold_sfr.shape[1]

46

## Part 5: Remove invalid records
#### 5.1 Remove duplicates

In [41]:
sold_sfr = sold_sfr.drop_duplicates(subset=['ListingKey', 'UnparsedAddress', 'ClosePrice', 'CloseDate'], keep='first')

In [ ]:
rows_before = cleaning_tracker.loc[step_i]['Rows After']
rows_after = sold_sfr.shape[0]
step_i = step_i + 1
cleaning_tracker.loc[step_i] = ['Dropped duplicates', rows_before, rows_after, (1 - rows_after / rows_before) * 100]

,Cleaning Step,Rows Before,Rows After,% Removed
0,Filtered for residential SFR,663761,334704,49.574621
1,Dropped duplicates,334704,334675,0.008664


#### 5.2 Remove records missing target

In [43]:
sold_sfr = sold_sfr.dropna(subset='ClosePrice')

In [ ]:
rows_before = cleaning_tracker.loc[step_i]['Rows After']
rows_after = sold_sfr.shape[0]
step_i = step_i + 1
cleaning_tracker.loc[step_i] = ['Dropped records without close price', rows_before, rows_after, (1 - rows_after / rows_before) * 100]

,Cleaning Step,Rows Before,Rows After,% Removed
0,Filtered for residential SFR,663761,334704,49.574621
1,Dropped duplicates,334704,334675,0.008664
2,Dropped records without close price,334675,334673,0.000598


#### 5.3 Remove records with logical inconsistencies
Focus on core features such as close price, living space configurations, sales and construction timeline, and geographical coordinates.

In [91]:
valid_price = (
    sold_sfr['ClosePrice'] > 0 & 
    (
        sold_sfr['AssociationFee'].isna() | 
        (sold_sfr['AssociationFee'] >= 0)
    )
)

In [92]:
valid_living_area = sold_sfr['LivingArea'].notna() & (sold_sfr['LivingArea'] > 0)
valid_beds = sold_sfr['BedroomsTotal'].isna() | (sold_sfr['BedroomsTotal'] >= 0)
valid_baths = sold_sfr['BathroomsTotalInteger'].isna() | (sold_sfr['BathroomsTotalInteger'] >= 0)

In [93]:
valid_lot = sold_sfr['LotSizeSquareFeet'].isna() | (sold_sfr['LotSizeSquareFeet'] >= 0)
valid_parking = sold_sfr['ParkingTotal'].isna() | (sold_sfr['ParkingTotal'] >= 0)
valid_stories = sold_sfr['Stories'].isna() | (sold_sfr['Stories'] >= 0)
valid_garage = (
    (sold_sfr['AttachedGarageYN'] == 'False') | 
    sold_sfr['AttachedGarageYN'].isna() |
    (sold_sfr['GarageSpaces'] >= 0)
)

In [94]:
valid_sales_cycle = (
    (sold_sfr['CloseDate'] >= sold_sfr['ListingContractDate']) &
    (
        sold_sfr['PurchaseContractDate'].isna() |
        sold_sfr['PurchaseContractDate'].between(
            sold_sfr['ListingContractDate'], 
            sold_sfr['CloseDate']
        )
    )
)

In [95]:
valid_built_year = (
    sold_sfr['YearBuilt'].isna() | 
    (
        (sold_sfr['YearBuilt'] <= sold_sfr['CloseDate'].dt.year) & 
        sold_sfr['YearBuilt'] > 0
    )
)

Records with incomplete coordinate information are considered invalid. Valid records must either have no coordinate information or have both latitude and longitude within the specified range.

In [96]:
ca_lat = [32.5, 43]
ca_lon = [-124.26, -114.8]
valid_coordinates = (
    (sold_sfr['Longitude'].isna() & sold_sfr['Latitude'].isna()) |
    (sold_sfr['Longitude'].between(ca_lon[0], ca_lon[1]) & sold_sfr['Latitude'].between(ca_lat[0], ca_lat[1]))
)

In [98]:
sold_sfr = sold_sfr[valid_price & 
                    valid_living_area & valid_beds & valid_baths & 
                    valid_lot & valid_garage & valid_parking & valid_stories &
                    valid_sales_cycle & valid_built_year &
                    valid_coordinates]

In [99]:
rows_before = cleaning_tracker.loc[step_i]['Rows After']
rows_after = sold_sfr.shape[0]
step_i = step_i + 1
cleaning_tracker.loc[step_i] = ['Dropped invalid records', rows_before, rows_after, (1 - rows_after / rows_before) * 100]

## Conclusion
The dataset has been restricted to the required property type, standardized, deduplidated, and cleaned of logically invalid records and features that could introduce target leakage. It's now ready for model development and testing.

The table below summarizes the row counts before and after each cleaning step.

In [100]:
cleaning_tracker

,Cleaning Step,Rows Before,Rows After,% Removed
0,Filtered for residential SFR,663761,334704,49.574621
1,Dropped duplicates,334704,334675,0.008664
2,Dropped records without close price,334675,334673,0.000598
3,Dropped invalid records,334673,328907,1.722876


Export the cleaned data for subsequent modeling.

In [101]:
sold_sfr.shape

(328907, 46)

In [102]:
sold_sfr.to_csv('clean-data/sold_clean.csv', index=False)